# `encode_python` — Compile Python Source Code

Pass Python source code that constructs a vector `f`. PyEncode either:
- **Path 1** (no hint): Parses the AST to recognise vectorised NumPy idioms
- **Path 2** (with `vector_type` hint): Executes the code and extracts parameters numerically

This makes it possible to handle for-loops, list comprehensions, and other constructs that the AST recogniser can't parse.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
import numpy as np
from qiskit.quantum_info import Statevector
from pyencode import encode_python

---
## DISCRETE — AST recognition (no hint needed)

The AST recogniser handles `np.zeros(N)` followed by a single indexed assignment.

In [ ]:
# Vectorised style: np.zeros + indexed assignment
code = """
N = 8
f = np.zeros(N)
f[3] = 5.0
"""
circuit, info = encode_python(code)
print(info)
circuit.draw('mpl')

In [ ]:
# For-loop style: AST can't parse this, so we provide a hint
code = """
import numpy as np
N = 8
f = np.zeros(N)
for i in range(N):
    if i == 3:
        f[i] = 5.0
"""
circuit, info = encode_python(code, vector_type="DISCRETE")
print(f"With hint: {info.vector_type}, gates={info.gate_count}")
circuit.draw('mpl')

---
## UNIFORM — AST recognition

In [ ]:
# np.ones idiom
circuit, info = encode_python("N = 8\nf = np.ones(N)")
print(info)
circuit.draw('mpl')

In [ ]:
# List comprehension style: needs hint
code = """
import numpy as np
N = 8
f = np.array([3.0 for _ in range(N)])
"""
circuit, info = encode_python(code, vector_type="UNIFORM")
print(f"List comp with hint: {info.vector_type}, gates={info.gate_count}")

---
## STEP — AST recognition

In [ ]:
# Slice assignment
circuit, info = encode_python("N = 8\nf = np.zeros(N)\nf[:4] = 1.0")
print(info)
circuit.draw('mpl')

---
## SINE — multiple coding styles

In [ ]:
# Vectorised np.sin with np.linspace: AST recognises this
code = "N = 16\nx = np.linspace(0, 1, N)\nf = np.sin(2*np.pi * x)"
circuit, info = encode_python(code)
print(f"Vectorised linspace: {info.vector_type}, gates={info.gate_count}")
circuit.draw('mpl')

In [ ]:
# For-loop: needs hint
code = """
import numpy as np
N = 16
f = np.zeros(N)
for i in range(N):
    f[i] = np.sin(2 * np.pi * i / N)
"""
circuit, info = encode_python(code, vector_type="SINE")
print(f"For-loop with hint: {info.vector_type}, n={info.params['n']}")

In [ ]:
# np.arange style: needs hint
code = """
import numpy as np
N = 16
k = np.arange(N)
f = np.sin(2 * np.pi * 3 * k / N)
"""
circuit, info = encode_python(code, vector_type="SINE")
print(f"arange style with hint: n={info.params['n']}")

---
## COSINE — AST recognition

In [ ]:
code = "N = 16\nx = np.linspace(0, 1, N)\nf = np.cos(2*np.pi * x)"
circuit, info = encode_python(code)
print(info)
circuit.draw('mpl')

---
## MULTI_DISCRETE — multiple assignments

In [ ]:
# Multiple indexed assignments: AST recognises this
code = "N = 8\nf = np.zeros(N)\nf[1] = 3.0\nf[6] = 4.0"
circuit, info = encode_python(code)
print(info)
circuit.draw('mpl')

In [ ]:
# zip-loop style: needs hint
code = """
import numpy as np
N = 8
indices = [1, 5, 7]
values = [2.0, 3.0, 1.0]
f = np.zeros(N)
for idx, val in zip(indices, values):
    f[idx] = val
"""
circuit, info = encode_python(code, vector_type="MULTI_DISCRETE")
print(f"zip-loop with hint: {info.vector_type}, gates={info.gate_count}")

---
## Shende fallback for unrecognised patterns

When no pattern is recognised and no hint is given, PyEncode falls back to Shende's general $\mathcal{O}(2^m)$ preparation and issues a warning.

In [ ]:
# Damped sine: not in the pattern library
code = """
import numpy as np
N = 8
x = np.linspace(0, 1, N)
f = np.sin(3 * 2*np.pi * x) * np.exp(-x)
"""

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    circuit, info = encode_python(code)
    print(f"Warning: {w[0].message}")

print(f"\nType: {info.vector_type}")
print(f"Complexity: {info.complexity}")
print(f"Gate count: {info.gate_count}")

In [ ]:
# Random vector: also falls back
code = """
import numpy as np
N = 4
f = np.array([0.5, 0.3, 0.7, 0.1])
"""

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    circuit, info = encode_python(code)

print(f"Type: {info.vector_type}, complexity: {info.complexity}")

---
## Callable input

`encode_python` also accepts a callable. The source is retrieved via `inspect.getsource`.

In [ ]:
def my_load():
    N = 8
    f = np.zeros(N)
    f[3] = 5.0

circuit, info = encode_python(my_load)
print(f"Callable: {info.vector_type}, gates={info.gate_count}")

---
## Circuit code output

Every call returns a standalone Python script in `info.circuit_code` that reproduces the circuit without PyEncode.

In [ ]:
code = "N = 8\nf = np.zeros(N)\nf[3] = 1.0"
_, info = encode_python(code)
print(info.circuit_code)